# Extraction

Sign in as the admin test user, download accepted PDFs from the private bucket, and run `pipeline.extractor.extract` once per parser tier and chunk tier. Parser tiers come from `config/parsers.py`. Chunk tiers come from `config/chunkers.py`: fast is one chunk per page, medium is the section chunker at 512 tokens, slow is the embedding tokenizer at 256 tokens.
Set `TIERS` and `CHUNK_TIERS_SELECTED` to the subsets you want. The run is the product of the two lists. Set `SHOW_TEXT = True` to print one chunk. The last cell deletes the temporary files.


In [ ]:
import os
import sys
import tempfile
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
os.chdir(ROOT)
for entry in (ROOT / "notebooks", ROOT):
    text = str(entry)
    if text not in sys.path:
        sys.path.insert(0, text)

load_dotenv(ROOT / ".env")

from helpers import download_supabase_pdfs, preview, timed

from config.chunkers import CHUNK_TIERS, chunk_method_for
from config.parsers import PARSER_TIERS, method_for

SHOW_TEXT = False
TIERS = list(PARSER_TIERS)
CHUNK_TIERS_SELECTED = list(CHUNK_TIERS)
workspace = tempfile.TemporaryDirectory(prefix="nb-extract-")

with timed("download") as setup:
    bucket, accepted, rejected = download_supabase_pdfs(Path(workspace.name))

print(bucket)
print(f"download: {setup['seconds']:.3f}s")
print(f"accepted: {len(accepted)}  rejected: {len(rejected)}")
for name, _size, reason in rejected:
    print(f"{name}\trejected: {reason}")
for tier in TIERS:
    print(f"parser\t{tier}\t{method_for(tier)}")
for tier in CHUNK_TIERS_SELECTED:
    print(f"chunk\t{tier}\t{chunk_method_for(tier)}")

In [ ]:
from pipeline.extractor import extract

documents = []
failures = []
for tier in TIERS:
    for chunk_tier in CHUNK_TIERS_SELECTED:
        chunk_method = chunk_method_for(chunk_tier)
        for path in accepted:
            try:
                with timed("extract") as timing:
                    document = await extract(
                        path,
                        tier=tier,
                        chunk_tier=chunk_tier,
                    )
                if document.metadata.total_chunks == 0:
                    failures.append(
                        (
                            tier,
                            chunk_tier,
                            path.name,
                            "RuntimeError",
                            "produced no text",
                        )
                    )
                    continue
                documents.append(
                    (tier, chunk_tier, chunk_method, document, timing["seconds"])
                )
            except Exception as exc:
                failures.append(
                    (tier, chunk_tier, path.name, type(exc).__name__, str(exc))
                )

In [ ]:
import statistics

header = (
    "parser\tchunk\tchunk_method\tfilename\tpages\tchunks"
    "\theadings\tmin\tmedian\tmax\tseconds"
)
print(header)
for tier, chunk_tier, chunk_method, document, seconds in documents:
    lengths = [len(chunk.text) for chunk in document.chunks]
    headings = {tuple(chunk.headings) for chunk in document.chunks if chunk.headings}
    print(
        tier,
        chunk_tier,
        chunk_method,
        document.metadata.filename,
        document.metadata.total_pages,
        document.metadata.total_chunks,
        len(headings),
        min(lengths),
        statistics.median(lengths),
        max(lengths),
        f"{seconds:.3f}",
        sep="\t",
    )

for tier, chunk_tier, name, exc_type, message in failures:
    print(f"{tier}\t{chunk_tier}\t{name}\tfailure\t{exc_type}\t{message}")

In [ ]:
if SHOW_TEXT and documents:
    tier, chunk_tier, _method, document, _seconds = documents[0]
    chunk = document.chunks[0]
    print(tier, chunk_tier, chunk.page, chunk.headings)
    print(preview(chunk.text, SHOW_TEXT))

In [ ]:
workspace.cleanup()